In [1]:
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import (
    cross_val_predict,
    cross_validate,
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

In [2]:
ROOT = Path.cwd().parent

RAW = ROOT / "data" / "processed" / "clean_old.csv"

df = pd.read_csv(RAW)

df["has_balance"] = (df["Balance"] > 0).astype(int)

df["age_group"] = pd.cut(
    df["Age"],
    bins=[17, 25, 35, 45, 55, 65, 75, 95],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76+"],
).astype(str)

In [3]:
X = df.drop(columns='Exited')

y = df['Exited']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [4]:
X_train_basic = X_train.drop(columns=["has_balance","age_group"])

In [5]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

### Dummy Baseline

In [6]:
dummy = DummyClassifier()

validation = cross_validate(
    dummy,
    X_train_basic,
    y_train,
    cv=cv,
    scoring="accuracy"
)

predictions = cross_val_predict(
    dummy,
    X_train_basic,
    y_train,
    cv=cv
)

print("Dummy baseline mean CV accuracy:", validation["test_score"].mean())

Dummy baseline mean CV accuracy: 0.79625


In [7]:
def report(name, model, X, y, cv):
    preds = cross_val_predict(model, X, y, cv=cv)
    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()

    print(name)
    print(pd.DataFrame(
        cm,
        index=["Actual stayed", "Actual churned"],
        columns=["Predicted stayed", "Predicted churned"]
    ))

    print(f"Accuracy:  {accuracy_score(y, preds):.3f}")
    print(f"Precision: {precision_score(y, preds, pos_label=1, zero_division=0):.3f}")
    print(f"Recall:    {recall_score(y, preds, pos_label=1, zero_division=0):.3f}")

    print(f"Caught: {tp}, Missed: {fn}, False alarms: {fp}")
    print()

#### Basic Logistic Regression with Original Features

In [8]:
preprocess = ColumnTransformer(
  [
    ("categorical",OneHotEncoder(),["Geography"]),
    ("num",StandardScaler(),["CreditScore","Age","Tenure","Balance","NumOfProducts","EstimatedSalary"]),
  ],
  remainder="passthrough"
)


pipe = Pipeline(
  [("preprocess",preprocess),
  ("model",LogisticRegression(max_iter=1000))]
)

validation = cross_validate(
  pipe,
  X_train_basic,
  y_train,
  cv=cv,
  scoring="accuracy",
  return_train_score=True
)

print("Basic Logistic mean CV accuracy:", validation["test_score"].mean())
print("Basic Logistic mean training accuracy:", validation["train_score"].mean())

report("Basic Logistic Regression", pipe, X_train_basic, y_train, cv)

Basic Logistic mean CV accuracy: 0.810375
Basic Logistic mean training accuracy: 0.8110937500000001
Basic Logistic Regression
                Predicted stayed  Predicted churned
Actual stayed               6135                235
Actual churned              1282                348
Accuracy:  0.810
Precision: 0.597
Recall:    0.213
Caught: 348, Missed: 1282, False alarms: 235



#### Logistic Regression with  NumOfProducts as a Categorical Feature

In [9]:
preprocess = ColumnTransformer(
    [
        ("category", OneHotEncoder(), ["Geography", "NumOfProducts"]),
        ("num", StandardScaler(), [
            "CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"
        ]),
    ],
    remainder="passthrough"
)

pipe = Pipeline(
    [
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

validation = cross_validate(
    pipe,
    X_train_basic,
    y_train,
    cv=cv,
    scoring="accuracy",
    return_train_score=True
)

print("Logistic model with NumOfProducts as category mean CV accuracy:",
      validation["test_score"].mean())

print("Mean training accuracy:", validation["train_score"].mean())

report("Logistic Regression with NumOfProducts as Categorical",
       pipe, X_train_basic, y_train, cv)

Logistic model with NumOfProducts as category mean CV accuracy: 0.8401249999999999
Mean training accuracy: 0.84040625
Logistic Regression with NumOfProducts as Categorical
                Predicted stayed  Predicted churned
Actual stayed               6115                255
Actual churned              1024                606
Accuracy:  0.840
Precision: 0.704
Recall:    0.372
Caught: 606, Missed: 1024, False alarms: 255



#### Logistic Regression with Has Balance Feature added

In [10]:
X_train_1 = X_train.drop(columns="age_group") 

preprocess = ColumnTransformer(
  [
    ("categorical",OneHotEncoder(),["Geography","NumOfProducts"]),
    ("num",StandardScaler(),["CreditScore","Age","Tenure","Balance","EstimatedSalary"]),
  ],
  remainder="passthrough"
)


pipe = Pipeline(
  [("preprocess",preprocess),
  ("model",LogisticRegression(max_iter=1000))]
)

validation = cross_validate(
  pipe,
  X_train_1,
  y_train,
  cv=cv,
  scoring="accuracy",
  return_train_score=True
)

print("Logistic model with has_balance CV mean accuracy:",validation['test_score'].mean())

print("Mean training accuracy:", validation["train_score"].mean())


report("Logistic Regression with Has_Balance", pipe, X_train_1, y_train, cv)



Logistic model with has_balance CV mean accuracy: 0.8398749999999999
Mean training accuracy: 0.84046875
Logistic Regression with Has_Balance
                Predicted stayed  Predicted churned
Actual stayed               6109                261
Actual churned              1020                610
Accuracy:  0.840
Precision: 0.700
Recall:    0.374
Caught: 610, Missed: 1020, False alarms: 261



#### Logistic Regression with Age Groups Feature + NumberOfProducts as categorical

In [11]:
X_train_1 = X_train.drop(columns="has_balance") 

preprocess = ColumnTransformer(
  [
    ("categorical",OneHotEncoder(),["Geography","NumOfProducts","age_group"]),
    ("num",StandardScaler(),["CreditScore","Age","Tenure","Balance","EstimatedSalary"]),
  ],
  remainder="passthrough"
)


pipe = Pipeline(
  [("preprocess",preprocess),
  ("model",LogisticRegression(max_iter=1000))]
)

validation = cross_validate(
  pipe,
  X_train_1,
  y_train,
  cv=cv,
  scoring="accuracy",
  return_train_score=True
)

print("Logistic with categorical NumOfProducts + age_group  mean CV accuracy:",validation['test_score'].mean())

print("Mean training accuracy:", validation["train_score"].mean())

report("Logistic Regression  Age Groups +NumOfProducts",
       pipe, X_train_1, y_train, cv)

Logistic with categorical NumOfProducts + age_group  mean CV accuracy: 0.851125
Mean training accuracy: 0.8526250000000001
Logistic Regression  Age Groups +NumOfProducts
                Predicted stayed  Predicted churned
Actual stayed               6079                291
Actual churned               900                730
Accuracy:  0.851
Precision: 0.715
Recall:    0.448
Caught: 730, Missed: 900, False alarms: 291



#### Logistic Regression with All Engineered Features

In [12]:
 

preprocess = ColumnTransformer(
  [
    ("category",OneHotEncoder(),["Geography","NumOfProducts","age_group"]),
    ("num",StandardScaler(),["CreditScore","Age","Tenure","Balance","EstimatedSalary"]),
  ],
  remainder="passthrough"
)


pipe = Pipeline(
  [("preprocess",preprocess),
  ("model",LogisticRegression(max_iter=1000))]
)

validation = cross_validate(
  pipe,
  X_train,
  y_train,
  cv=cv,
  scoring="accuracy",
  return_train_score=True
)

print("Logistic with all engineered features  mean CV accuracy:",
      validation["test_score"].mean())

print("Mean training accuracy:", validation["train_score"].mean())

report("Logistic Regression with  all engineered Features",
       pipe, X_train, y_train, cv)

Logistic with all engineered features  mean CV accuracy: 0.851
Mean training accuracy: 0.8530312499999999
Logistic Regression with  all engineered Features
                Predicted stayed  Predicted churned
Actual stayed               6075                295
Actual churned               897                733
Accuracy:  0.851
Precision: 0.713
Recall:    0.450
Caught: 733, Missed: 897, False alarms: 295



####  Tuned Logistic Regression using GridSearchCV

In [13]:


preprocess = ColumnTransformer(
  [
    ("category", OneHotEncoder(), ["Geography", "NumOfProducts", "age_group"]),
    ("num", StandardScaler(), ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]),
  ],
  remainder="passthrough"
)

pipe = Pipeline([
  ("preprocess", preprocess),
  ("model", LogisticRegression(max_iter=1000))
])



param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight":[None,"balanced"],
}


grid = GridSearchCV(
  estimator=pipe,
  param_grid=param_grid,
  cv=cv,
  scoring="roc_auc",
  return_train_score=True
)

grid.fit(X_train,y_train)

print("Best parameters:", grid.best_params_)
print(f"Best mean CV ROC-AUC: {grid.best_score_:.3f}")
print(f"Mean training ROC-AUC: {grid.cv_results_['mean_train_score'][grid.best_index_]:.3f}")

report("Tuned Logistic Regression", grid.best_estimator_, X_train, y_train, cv)

Best parameters: {'model__C': 10, 'model__class_weight': 'balanced'}
Best mean CV ROC-AUC: 0.848
Mean training ROC-AUC: 0.851
Tuned Logistic Regression
                Predicted stayed  Predicted churned
Actual stayed               5035               1335
Actual churned               410               1220
Accuracy:  0.782
Precision: 0.477
Recall:    0.748
Caught: 1220, Missed: 410, False alarms: 1335



## Logistic Regression — Results

| Model | Accuracy | Precision | Recall |
|---|---:|---:|---:|
| Basic Logistic | 81.04% | 59.7% | 21.3% |
| NumOfProducts | 84.01% | 70.4% | 37.2% |
| Has Balance | 83.99% | 70.0% | 37.4% |
| Age Groups + NumOfProducts | 85.11% | 71.5% | 44.8% |
| All Engineered Features | 85.10% | 71.3% | 45.0% |
| Tuned Logistic (Balanced) | 78.19% | 47.7% | 74.8% |

**Tuned model:** C = 10, class_weight = balanced

**Mean CV ROC-AUC:** 0.848

### Key Observations

Treating NumOfProducts as a category improved recall from 21% to 37%, while adding age groups increased it further to 45%. The has_balance flag had minimal impact. Changing C made no difference, with AUC remaining at 0.848 for values of 1 or higher. Similar training and validation scores indicate no overfitting. Balanced class weights increased recall to 75% but reduced precision to 48%. 
It simply makes the model more likely to predict positive cases, increasing recall but reducing precision without improving overall performance.

## Baseline Decision Tree

In [14]:
preprocess_tree = ColumnTransformer(
    [
        ("categorical", OneHotEncoder(), ["Geography", "NumOfProducts"]),
    ],
    remainder="passthrough"
)

tree_pipe = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", DecisionTreeClassifier(random_state=42))
])

validation = cross_validate(
    tree_pipe,
    X_train_basic,
    y_train,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True
)

print("Baseline Decision Tree mean CV ROC-AUC:", validation["test_score"].mean())

print("Baseline Decision mean training ROC-AUC:", validation["train_score"].mean())

report("Baseline Decision Tree", tree_pipe, X_train_basic, y_train, cv)

Baseline Decision Tree mean CV ROC-AUC: 0.6804263659215456
Baseline Decision mean training ROC-AUC: 1.0
Baseline Decision Tree
                Predicted stayed  Predicted churned
Actual stayed               5468                902
Actual churned               811                819
Accuracy:  0.786
Precision: 0.476
Recall:    0.502
Caught: 819, Missed: 811, False alarms: 902



## Decision Tree with Hyperparameter Tuning

In [15]:
param_grid = {
    "model__max_depth": [3, 5, 10, 15, None],
    "model__min_samples_split": [2, 10, 20],
    "model__min_samples_leaf": [1, 5, 10],
    "model__criterion": ["gini", "entropy"],
    "model__ccp_alpha": [0.0, 0.001, 0.01],
    "model__max_leaf_nodes": [10, 20, 50, None],
    "model__class_weight": [None, "balanced"]
}

tree_grid = GridSearchCV(
    estimator=tree_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    n_jobs=2
)

tree_grid.fit(X_train_basic, y_train)

print("Best parameters:", tree_grid.best_params_)
print("Best mean CV ROC-AUC:", tree_grid.best_score_)

best_tree = tree_grid.best_estimator_

report(
    "Best Tuned Decision Tree",
    best_tree,
    X_train_basic,
    y_train,
    cv
)

Best parameters: {'model__ccp_alpha': 0.0, 'model__class_weight': 'balanced', 'model__criterion': 'entropy', 'model__max_depth': 10, 'model__max_leaf_nodes': 20, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
Best mean CV ROC-AUC: 0.8387389122709017
Best Tuned Decision Tree
                Predicted stayed  Predicted churned
Actual stayed               4809               1561
Actual churned               364               1266
Accuracy:  0.759
Precision: 0.448
Recall:    0.777
Caught: 1266, Missed: 364, False alarms: 1561



In [16]:
print(
    "Best mean training ROC-AUC:",
    tree_grid.cv_results_["mean_train_score"][tree_grid.best_index_]
)

Best mean training ROC-AUC: 0.8542820936184761


## Decision Tree: Results

| Model | Accuracy | Precision | Recall | CV ROC-AUC |
|---|---:|---:|---:|---:|
| Baseline tree | 78.6% | 47.6% | 50.2% | 0.680 |
| Tuned tree | 75.9% | 44.8% | 77.7% | 0.839 |


**Observations**
 The untuned tree overfitted. It achieved a training AUC of 1.0 but only 0.680 on validation. Limiting its size reduced overfitting, bringing the scores closer at 0.854 and 0.839. Balanced class weights increased recall to 77.7% but reduced precision and accuracy. Its AUC remains lower than logistic regression at 0.848

## Baseline Random Forest

In [17]:
preprocess_tree = ColumnTransformer(
    [
        ("cat", OneHotEncoder(), ["Geography", "NumOfProducts"]),
    ],
    remainder="passthrough"
)

rf_pipe = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=1
    ))
])

validation = cross_validate(
    rf_pipe,
    X_train_basic,
    y_train,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True
)

report("Basic Random Forest", rf_pipe, X_train_basic, y_train, cv)

print("Random Forest mean CV ROC-AUC:", validation["test_score"].mean())
print("Random Forest mean training ROC-AUC:", validation["train_score"].mean())

Basic Random Forest
                Predicted stayed  Predicted churned
Actual stayed               6120                250
Actual churned               870                760
Accuracy:  0.860
Precision: 0.752
Recall:    0.466
Caught: 760, Missed: 870, False alarms: 250

Random Forest mean CV ROC-AUC: 0.8493872253951131
Random Forest mean training ROC-AUC: 1.0


### Random Forest  Hyperparameter Tuning

In [18]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_leaf": [1, 5],
    "model__max_features": ["sqrt"],
    "model__class_weight": [None, "balanced"]
}

rf_grid = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    n_jobs=2
)

rf_grid.fit(X_train_basic, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best mean CV ROC-AUC:", rf_grid.best_score_)

best_rf = rf_grid.best_estimator_

report("Best Tuned Random Forest", best_rf, X_train_basic, y_train, cv)

i = rf_grid.best_index_

print("Best mean training ROC-AUC:", rf_grid.cv_results_["mean_train_score"][i])
print("Best mean validation ROC-AUC:", rf_grid.best_score_)

Best parameters: {'model__class_weight': None, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 200}
Best mean CV ROC-AUC: 0.8602575338771657
Best Tuned Random Forest
                Predicted stayed  Predicted churned
Actual stayed               6189                181
Actual churned               927                703
Accuracy:  0.862
Precision: 0.795
Recall:    0.431
Caught: 703, Missed: 927, False alarms: 181

Best mean training ROC-AUC: 0.937507132985332
Best mean validation ROC-AUC: 0.8602575338771657


## Random Forest: Results

| Model | Accuracy | Precision | Recall | CV ROC-AUC |
|---|---:|---:|---:|---:|
| Baseline forest | 86.0% | 75.2% | 46.6% | 0.849 |
| Tuned forest | 86.2% | 79.5% | 43.1% | 0.860 |


**Observations** 
Tuning the randpm improved AUC from 0.849 to 0.860 and precision from 75% to 80%, but reduced recall from 47% to 43%. The tuned forest is more cautious when flagging churners. Some overfitting remains, with training AUC at 0.938 compared to 0.860 on validation. Lowering the threshold can help increase recall.

##Baseline XGBOOST

In [19]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=1
)

xgb_pipe = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", xgb_model)
])



validation = cross_validate(
    xgb_pipe,
    X_train_basic,
    y_train,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True
)
 

print("XGBoost mean CV ROC-AUC:", validation["test_score"].mean())
print("XGBoost mean training ROC-AUC:", validation["train_score"].mean())

report("Basic XGBoost", xgb_pipe, X_train_basic, y_train, cv)

XGBoost mean CV ROC-AUC: 0.8425340216313048
XGBoost mean training ROC-AUC: 0.9972597899471255
Basic XGBoost
                Predicted stayed  Predicted churned
Actual stayed               6019                351
Actual churned               829                801
Accuracy:  0.853
Precision: 0.695
Recall:    0.491
Caught: 801, Missed: 829, False alarms: 351



###  XGBoost  with Hyperparameter Tuning

In [20]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.05, 0.1],
    "model__min_child_weight": [1, 5],
    "model__scale_pos_weight": [1, 3.9]
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    n_jobs=2
)

xgb_grid.fit(X_train_basic, y_train)

print("Best parameters:", xgb_grid.best_params_)
print("Best mean CV ROC-AUC:", xgb_grid.best_score_)

best_index = xgb_grid.best_index_

print("Best mean training ROC-AUC:", xgb_grid.cv_results_["mean_train_score"][best_index])
print("Best mean validation ROC-AUC:", xgb_grid.cv_results_["mean_test_score"][best_index])

best_xgb = xgb_grid.best_estimator_

report(
    "Best Tuned XGBoost",
    best_xgb,
    X_train_basic,
    y_train,
    cv
)

Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__min_child_weight': 5, 'model__n_estimators': 200, 'model__scale_pos_weight': 1}
Best mean CV ROC-AUC: 0.8662388881933142
Best mean training ROC-AUC: 0.9072816644354769
Best mean validation ROC-AUC: 0.8662388881933142
Best Tuned XGBoost
                Predicted stayed  Predicted churned
Actual stayed               6120                250
Actual churned               848                782
Accuracy:  0.863
Precision: 0.758
Recall:    0.480
Caught: 782, Missed: 848, False alarms: 250



## XGBoost: Results

| Model | Accuracy | Precision | Recall | CV ROC-AUC |
|---|---:|---:|---:|---:|
| Baseline XGBoost | 85.3% | 69.5% | 49.1% | 0.843 |
| Tuned XGBoost | 86.3% | 75.8% | 48.0% | 0.866 |



**Observations** 
The Tuned XGBOOST improved AUC from 0.843 to 0.866 and precision from 69.5% to 75.8%, while recall dropped slightly at the default 0.5 threshold. Limiting tree depth to 3 reduced overfitting, with training AUC at 0.907 and validation AUC at 0.866. It ranks churners better than the other models, and recall can be improved by lowering the threshold.

## Model Comparison

## Model Comparison

Logistic regression used engineered features, while the other three models used the original 10 columns. Logistic regression and the decision tree used balanced class weights which resulted in higher recall but lower precision.

| Model               | Accuracy | Precision | Recall | CV ROC-AUC |
| ------------------- | -------- | --------- | ------ | ---------- |
| Logistic Regression | 78.2%    | 47.7%     | 74.8%  | 0.848      |
| Decision Tree       | 75.9%    | 44.8%     | 77.7%  | 0.839      |
| Random Forest       | 86.2%    | 79.5%     | 43.1%  | 0.860      |
| XGBoost             | 86.3%    | 75.8%     | 48.0%  | 0.866      |

**Observations**
XGBoost achieved the highest AUC at 0.866, closely followed by the random forest at 0.860. Logistic regression also performed well, with an AUC of 0.848. At similar recall, XGBoost was slightly more precise than logistic regression.

Selection: XGBoost is the preferred model because it has the highest AUC and less overfitting than the random forest. The next step is to select an appropriate classification threshold.

## XGBoost with Threshold Tuning

In [21]:
xgb_probs = cross_val_predict(
    best_xgb,
    X_train_basic,
    y_train,
    cv=cv,
    method="predict_proba"
)[:, 1]

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30,
              0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

for threshold in thresholds:
    predictions = (xgb_probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_train, predictions).ravel()

    precision = precision_score(y_train, predictions)
    recall = recall_score(y_train, predictions)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"False alarms: {fp}")
    print(f"Churners missed: {fn}")


Threshold: 0.05
Precision: 0.284
Recall: 0.956
False alarms: 3930
Churners missed: 72

Threshold: 0.1
Precision: 0.372
Recall: 0.886
False alarms: 2437
Churners missed: 186

Threshold: 0.15
Precision: 0.443
Recall: 0.821
False alarms: 1681
Churners missed: 292

Threshold: 0.2
Precision: 0.500
Recall: 0.766
False alarms: 1248
Churners missed: 381

Threshold: 0.25
Precision: 0.550
Recall: 0.709
False alarms: 945
Churners missed: 474

Threshold: 0.3
Precision: 0.606
Recall: 0.663
False alarms: 704
Churners missed: 549

Threshold: 0.35
Precision: 0.640
Recall: 0.612
False alarms: 561
Churners missed: 632

Threshold: 0.4
Precision: 0.677
Recall: 0.565
False alarms: 440
Churners missed: 709

Threshold: 0.45
Precision: 0.723
Recall: 0.525
False alarms: 328
Churners missed: 774

Threshold: 0.5
Precision: 0.758
Recall: 0.480
False alarms: 250
Churners missed: 848

Threshold: 0.55
Precision: 0.792
Recall: 0.441
False alarms: 189
Churners missed: 911

Threshold: 0.6
Precision: 0.826
Recall: 0.40

### XGBoost  Threshold Selection

| Threshold | Precision | Recall | False Alarms | Churners Missed |
| --------- | --------- | ------ | ------------ | --------------- |
| 0.10      | 37.2%     | 88.6%  | 2,437        | 186             |
| 0.15      | 44.3%     | 82.1%  | 1,681        | 292             |
| 0.20      | 50.0%     | 76.6%  | 1,248        | 381             |
| 0.25      | 55.0%     | 70.9%  | 945          | 474             |
| 0.30      | 60.6%     | 66.3%  | 704          | 549             |
| 0.50      | 75.8%     | 48.0%  | 250          | 848             |

### Selected Threshold: 0.15

0.15 was  because the main goal is to identify as many potential churners as possible. At this threshold, the model catches 82.1% of churners, compared with 48.0% at the default threshold of 0.50. Although this increases false alarms, it reduces missed churners and gives the business more opportunities to retain customers before they leave.

## Final Test on the X_test

In [ ]:
THRESHOLD = 0.15  

X_test_basic = X_test.drop(columns=["has_balance", "age_group"])

test_probs = best_xgb.predict_proba(X_test_basic)[:, 1]
test_predictions = (test_probs >= THRESHOLD).astype(int)

print("AUC:", roc_auc_score(y_test, test_probs))

print(confusion_matrix(y_test, test_predictions))
print("Precision:", precision_score(y_test, test_predictions))

print("Recall:", recall_score(y_test, test_predictions))

AUC: 0.8673650538057316
[[1144  449]
 [  67  340]]
Precision: 0.4309252217997465
Recall: 0.8353808353808354


**Result:** On 2,000 unseen customers, including 407 churners, the model achieved an ROC-AUC of 0.867, recall of 83.5% and precision of 43.1%. It correctly identified 340 churners, missed 67 and produced 449 false alarms. The results closely match cross-validation